# 03 — Data Cleaning

Based on the findings in notebook 02, this notebook applies a documented, reproducible
cleaning pipeline (implemented in `src/preprocessing.py`, not copy-pasted here) and saves the
cleaned dataset for all downstream notebooks to reuse.

**Decision: 40% missing-value drop threshold.**
Notebook 02 showed a clean gap — 32 sensors missing on 46–91% of units, and every other
sensor missing on 17% or less. A 40% threshold sits inside that gap, so the choice isn't
sensitive to the exact cutoff value. See `docs/decision_log.md` for the full write-up.


In [1]:

import sys
sys.path.insert(0, "../src")
import pandas as pd
import preprocessing as prep

df = prep.load_raw_data("../data/raw/uci-secom.csv")
result = prep.clean_pipeline(df, threshold_pct=40.0)

print("Cleaning summary:")
for k, v in result['summary'].items():
    print(f"  {k}: {v}")


Cleaning summary:
  total_units: 1567
  pass_units: 1463
  fail_units: 104
  raw_sensors: 590
  dropped_high_missing: 32
  dropped_zero_variance: 116
  cleaned_sensors: 442
  missing_threshold_pct: 40.0
  fail_rate_pct: 6.64


In [2]:

print(f"Dropped for >40% missing ({len(result['dropped_missing'])} sensors):")
print(result['dropped_missing'])
print()
print(f"Dropped for zero variance ({len(result['dropped_variance'])} sensors), first 20 shown:")
print(result['dropped_variance'][:20], '...')


Dropped for >40% missing (32 sensors):
['72', '73', '85', '109', '110', '111', '112', '157', '158', '220', '244', '245', '246', '247', '292', '293', '345', '346', '358', '382', '383', '384', '385', '492', '516', '517', '518', '519', '578', '579', '580', '581']

Dropped for zero variance (116 sensors), first 20 shown:
['5', '13', '42', '49', '52', '69', '97', '141', '149', '178', '179', '186', '189', '190', '191', '192', '193', '194', '226', '229'] ...


## Remaining missing values: median imputation

**Decision: median imputation, not mean.**
Sensor readings can contain outliers/skew (equipment glitches), and the median is robust to
that in a way the mean is not. Since imputation only touches sensors already below the 40%
missing threshold, the number of imputed values per column is small relative to the 1,567
units. See `docs/decision_log.md`.


In [3]:

cleaned = result['data']
remaining_na = cleaned[result['cleaned_sensor_cols']].isna().sum().sum()
print("Remaining NaNs after cleaning pipeline (should be 0, median-imputed):", remaining_na)
print("Final cleaned shape:", cleaned.shape)

cleaned.to_csv("../data/processed/secom_cleaned.csv", index=False)
print("Saved -> data/processed/secom_cleaned.csv")


Remaining NaNs after cleaning pipeline (should be 0, median-imputed): 0
Final cleaned shape: (1567, 444)
Saved -> data/processed/secom_cleaned.csv


## Cleaning pipeline summary

```
590 raw sensors
   │
   ▼  drop >40% missing (32 sensors)
558 sensors
   │
   ▼  drop zero-variance (116 sensors)
442 sensors
   │
   ▼  median-impute remaining missing values
442 clean, complete sensors  →  data/processed/secom_cleaned.csv
```

This is the dataset every later notebook (EDA, feature engineering, modeling, validation)
loads — nothing downstream re-derives cleaning rules, keeping the pipeline single-sourced.